In [2]:
import pandas as pd  # WHY: used for loading and manipulating tabular data

df = pd.read_csv("C:\sujit\SEM_5\SMART_GRID\datasets\energy_dataset.csv")  # WHY: load the raw hourly energy dataset

print(df.shape)  # WHY: verify number of rows and columns
print(df.head())  # WHY: visually inspect the first few observations

print(df.info())  # WHY: inspect column types and non-null counts
print(df.isnull().sum())  # WHY: identify missing values in each column

print("Duplicates:", df.duplicated().sum())  # WHY: detect repeated rows

(35064, 29)
                        time  generation biomass  \
0  2015-01-01 00:00:00+01:00               447.0   
1  2015-01-01 01:00:00+01:00               449.0   
2  2015-01-01 02:00:00+01:00               448.0   
3  2015-01-01 03:00:00+01:00               438.0   
4  2015-01-01 04:00:00+01:00               428.0   

   generation fossil brown coal/lignite  generation fossil coal-derived gas  \
0                                 329.0                                 0.0   
1                                 328.0                                 0.0   
2                                 323.0                                 0.0   
3                                 254.0                                 0.0   
4                                 187.0                                 0.0   

   generation fossil gas  generation fossil hard coal  generation fossil oil  \
0                 4844.0                       4821.0                  162.0   
1                 5196.0                

In [3]:
empty_columns = [  # WHY: these columns contain no usable observations
    "generation hydro pumped storage aggregated",
    "forecast wind offshore eday ahead",
]

df = df.drop(columns=empty_columns)  # WHY: empty features cannot help prediction

print(df.shape)  # WHY: confirm that exactly two columns were removed
print(df.columns.tolist())  # WHY: verify the remaining feature names

(35064, 27)
['time', 'generation biomass', 'generation fossil brown coal/lignite', 'generation fossil coal-derived gas', 'generation fossil gas', 'generation fossil hard coal', 'generation fossil oil', 'generation fossil oil shale', 'generation fossil peat', 'generation geothermal', 'generation hydro pumped storage consumption', 'generation hydro run-of-river and poundage', 'generation hydro water reservoir', 'generation marine', 'generation nuclear', 'generation other', 'generation other renewable', 'generation solar', 'generation waste', 'generation wind offshore', 'generation wind onshore', 'forecast solar day ahead', 'forecast wind onshore day ahead', 'total load forecast', 'total load actual', 'price day ahead', 'price actual']


In [4]:
df["time"] = pd.to_datetime(  # WHY: convert timestamp strings into real datetime values
    df["time"],
    utc=True,  # WHY: normalize timezone-aware timestamps consistently
)

df = df.sort_values("time")  # WHY: forecasting requires strict chronological order
df = df.reset_index(drop=True)  # WHY: restore sequential row indices after sorting

print(df["time"].dtype)  # WHY: confirm datetime conversion worked
print(df[["time"]].head())  # WHY: verify chronological ordering

datetime64[us, UTC]
                       time
0 2014-12-31 23:00:00+00:00
1 2015-01-01 00:00:00+00:00
2 2015-01-01 01:00:00+00:00
3 2015-01-01 02:00:00+00:00
4 2015-01-01 03:00:00+00:00


In [5]:
missing = df.isnull().sum()  # WHY: count missing observations in every column

missing = missing[missing > 0]  # WHY: keep only columns that actually have missing data

print(missing)  # WHY: see exactly what still needs cleaning

generation biomass                             19
generation fossil brown coal/lignite           18
generation fossil coal-derived gas             18
generation fossil gas                          18
generation fossil hard coal                    18
generation fossil oil                          19
generation fossil oil shale                    18
generation fossil peat                         18
generation geothermal                          18
generation hydro pumped storage consumption    19
generation hydro run-of-river and poundage     19
generation hydro water reservoir               18
generation marine                              19
generation nuclear                             17
generation other                               18
generation other renewable                     18
generation solar                               18
generation waste                               19
generation wind offshore                       18
generation wind onshore                        18


In [6]:
numeric_cols = df.select_dtypes(include="number").columns  # WHY: interpolate only numerical data

feature_cols = numeric_cols.drop("price actual")  # WHY: never manufacture our prediction target

df[feature_cols] = df[feature_cols].interpolate(
    method="linear",  # WHY: estimate a missing hourly value from neighboring observations
    limit_direction="both"  # WHY: also handles a gap at either dataset boundary
)

print("Missing values remaining:")
print(df.isnull().sum()[df.isnull().sum() > 0])  # WHY: verify cleaning succeeded

Missing values remaining:
Series([], dtype: int64)


In [7]:
print("Shape:", df.shape)  # WHY: ensure we didn't accidentally delete observations
print("Missing:", df.isnull().sum().sum())  # WHY: should now be zero
print("Duplicates:", df.duplicated().sum())  # WHY: confirm rows remain unique
print(df.head())  # WHY: visually verify the cleaned dataset

Shape: (35064, 27)
Missing: 0
Duplicates: 0
                       time  generation biomass  \
0 2014-12-31 23:00:00+00:00               447.0   
1 2015-01-01 00:00:00+00:00               449.0   
2 2015-01-01 01:00:00+00:00               448.0   
3 2015-01-01 02:00:00+00:00               438.0   
4 2015-01-01 03:00:00+00:00               428.0   

   generation fossil brown coal/lignite  generation fossil coal-derived gas  \
0                                 329.0                                 0.0   
1                                 328.0                                 0.0   
2                                 323.0                                 0.0   
3                                 254.0                                 0.0   
4                                 187.0                                 0.0   

   generation fossil gas  generation fossil hard coal  generation fossil oil  \
0                 4844.0                       4821.0                  162.0   
1             

In [8]:
df["time"] = df["time"].dt.tz_convert(
    "Europe/Madrid"
)  # WHY: price/load behaviour depends on the actual local market hour

print(df["time"].head())  # WHY: verify local timestamps were restored
print(df["time"].tail())  # WHY: check the dataset ending timestamps too

0   2015-01-01 00:00:00+01:00
1   2015-01-01 01:00:00+01:00
2   2015-01-01 02:00:00+01:00
3   2015-01-01 03:00:00+01:00
4   2015-01-01 04:00:00+01:00
Name: time, dtype: datetime64[us, Europe/Madrid]
35059   2018-12-31 19:00:00+01:00
35060   2018-12-31 20:00:00+01:00
35061   2018-12-31 21:00:00+01:00
35062   2018-12-31 22:00:00+01:00
35063   2018-12-31 23:00:00+01:00
Name: time, dtype: datetime64[us, Europe/Madrid]


In [9]:
df["hour"] = df["time"].dt.hour  # WHY: electricity demand and price follow daily cycles

df["day_of_week"] = df["time"].dt.dayofweek  # WHY: weekdays behave differently

df["month"] = df["time"].dt.month  # WHY: electricity behaviour changes seasonally

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)  # WHY: Saturday/Sunday demand patterns differ from weekdays

print(df[["time", "hour", "day_of_week", "month", "is_weekend"]].head(10))

                       time  hour  day_of_week  month  is_weekend
0 2015-01-01 00:00:00+01:00     0            3      1           0
1 2015-01-01 01:00:00+01:00     1            3      1           0
2 2015-01-01 02:00:00+01:00     2            3      1           0
3 2015-01-01 03:00:00+01:00     3            3      1           0
4 2015-01-01 04:00:00+01:00     4            3      1           0
5 2015-01-01 05:00:00+01:00     5            3      1           0
6 2015-01-01 06:00:00+01:00     6            3      1           0
7 2015-01-01 07:00:00+01:00     7            3      1           0
8 2015-01-01 08:00:00+01:00     8            3      1           0
9 2015-01-01 09:00:00+01:00     9            3      1           0


In [10]:
print("New shape:", df.shape)  # WHY: confirm four engineered features were added

print(df[
    ["time", "hour", "day_of_week", "month", "is_weekend", "price actual"]
].head())  # WHY: visually verify the engineered features

print(df[
    ["hour", "day_of_week", "month", "is_weekend"]
].describe())  # WHY: check whether feature ranges make sense

New shape: (35064, 31)
                       time  hour  day_of_week  month  is_weekend  \
0 2015-01-01 00:00:00+01:00     0            3      1           0   
1 2015-01-01 01:00:00+01:00     1            3      1           0   
2 2015-01-01 02:00:00+01:00     2            3      1           0   
3 2015-01-01 03:00:00+01:00     3            3      1           0   
4 2015-01-01 04:00:00+01:00     4            3      1           0   

   price actual  
0         65.41  
1         64.92  
2         64.48  
3         59.32  
4         56.04  
               hour   day_of_week         month    is_weekend
count  35064.000000  35064.000000  35064.000000  35064.000000
mean      11.500000      3.002053      6.523728      0.286105
std        6.922285      2.000541      3.448747      0.451946
min        0.000000      0.000000      1.000000      0.000000
25%        5.750000      1.000000      4.000000      0.000000
50%       11.500000      3.000000      7.000000      0.000000
75%       17.250000 

In [11]:
price_lags = [1, 2, 3, 24, 48, 168]  # WHY: capture short, daily, and weekly price patterns

for lag in price_lags:
    df[f"price_lag_{lag}"] = df["price actual"].shift(lag)  # WHY: use only past prices

print(df[
    ["time", "price actual", "price_lag_1", "price_lag_24", "price_lag_168"]
].head(170))  # WHY: verify lag alignment

                         time  price actual  price_lag_1  price_lag_24  \
0   2015-01-01 00:00:00+01:00         65.41          NaN           NaN   
1   2015-01-01 01:00:00+01:00         64.92        65.41           NaN   
2   2015-01-01 02:00:00+01:00         64.48        64.92           NaN   
3   2015-01-01 03:00:00+01:00         59.32        64.48           NaN   
4   2015-01-01 04:00:00+01:00         56.04        59.32           NaN   
..                        ...           ...          ...           ...   
165 2015-01-07 21:00:00+01:00         87.08        91.25         78.63   
166 2015-01-07 22:00:00+01:00         78.15        87.08         73.20   
167 2015-01-07 23:00:00+01:00         75.07        78.15         72.46   
168 2015-01-08 00:00:00+01:00         73.73        75.07         67.24   
169 2015-01-08 01:00:00+01:00         70.99        73.73         64.17   

     price_lag_168  
0              NaN  
1              NaN  
2              NaN  
3              NaN  
4     

In [12]:
load_lags = [1, 24, 168]  # WHY: capture recent, daily, and weekly demand behaviour

for lag in load_lags:
    df[f"load_lag_{lag}"] = df["total load actual"].shift(lag)  # WHY: use historical load only

print(df[
    ["time", "total load actual", "load_lag_1", "load_lag_24", "load_lag_168"]
].head(170))  # WHY: verify load lags were created correctly

                         time  total load actual  load_lag_1  load_lag_24  \
0   2015-01-01 00:00:00+01:00            25385.0         NaN          NaN   
1   2015-01-01 01:00:00+01:00            24382.0     25385.0          NaN   
2   2015-01-01 02:00:00+01:00            22734.0     24382.0          NaN   
3   2015-01-01 03:00:00+01:00            21286.0     22734.0          NaN   
4   2015-01-01 04:00:00+01:00            20264.0     21286.0          NaN   
..                        ...                ...         ...          ...   
165 2015-01-07 21:00:00+01:00            33716.0     34470.0      30747.0   
166 2015-01-07 22:00:00+01:00            32697.0     33716.0      29978.0   
167 2015-01-07 23:00:00+01:00            30477.0     32697.0      27421.0   
168 2015-01-08 00:00:00+01:00            26788.0     30477.0      30518.0   
169 2015-01-08 01:00:00+01:00            25146.0     26788.0      28484.0   

     load_lag_168  
0             NaN  
1             NaN  
2             N

In [13]:
df["price_mean_24"] = df["price actual"].shift(1).rolling(
    window=24
).mean()  # WHY: summarize the previous 24 hours without including current price

df["price_std_24"] = df["price actual"].shift(1).rolling(
    window=24
).std()  # WHY: measure recent price volatility

df["load_mean_24"] = df["total load actual"].shift(1).rolling(
    window=24
).mean()  # WHY: summarize recent electricity demand

print(df[
    ["time", "price actual", "price_mean_24",
     "price_std_24", "load_mean_24"]
].head(30))  # WHY: verify rolling values begin after sufficient history

                        time  price actual  price_mean_24  price_std_24  \
0  2015-01-01 00:00:00+01:00         65.41            NaN           NaN   
1  2015-01-01 01:00:00+01:00         64.92            NaN           NaN   
2  2015-01-01 02:00:00+01:00         64.48            NaN           NaN   
3  2015-01-01 03:00:00+01:00         59.32            NaN           NaN   
4  2015-01-01 04:00:00+01:00         56.04            NaN           NaN   
5  2015-01-01 05:00:00+01:00         53.63            NaN           NaN   
6  2015-01-01 06:00:00+01:00         51.73            NaN           NaN   
7  2015-01-01 07:00:00+01:00         51.43            NaN           NaN   
8  2015-01-01 08:00:00+01:00         48.98            NaN           NaN   
9  2015-01-01 09:00:00+01:00         54.20            NaN           NaN   
10 2015-01-01 10:00:00+01:00         58.94            NaN           NaN   
11 2015-01-01 11:00:00+01:00         59.86            NaN           NaN   
12 2015-01-01 12:00:00+01

In [14]:
lagged_df = df.dropna().copy()  # WHY: remove only rows lacking required history

lagged_df = lagged_df.reset_index(
    drop=True
)  # WHY: restore clean sequential indexing after row removal

print("Before:", df.shape)  # WHY: compare original engineered dataset size
print("After:", lagged_df.shape)  # WHY: confirm only initial history rows were lost
print("Missing:", lagged_df.isnull().sum().sum())  # WHY: should be zero

Before: (35064, 43)
After: (34896, 43)
Missing: 0


In [15]:
features = [
    "hour", "day_of_week", "month", "is_weekend",
    "price day ahead", "total load forecast",
    "forecast solar day ahead", "forecast wind onshore day ahead",
    "price_lag_1", "price_lag_2", "price_lag_3",
    "price_lag_24", "price_lag_48", "price_lag_168",
    "load_lag_1", "load_lag_24", "load_lag_168",
    "price_mean_24", "price_std_24", "load_mean_24"
]  # WHY: only use information available before the target hour

X = lagged_df[features].copy()  # WHY: predictor variables
y = lagged_df["price actual"].copy()  # WHY: actual electricity price is our target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (34896, 20)
y shape: (34896,)


In [16]:
n = len(X)  # WHY: determine split boundaries from total chronological samples
train_end = int(n * 0.70)  # WHY: first 70% is used for learning
val_end = int(n * 0.85)  # WHY: next 15% is used for model selection

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (24427, 20) (24427,)
Validation: (5234, 20) (5234,)
Test: (5235, 20) (5235,)


In [17]:
import numpy as np  # WHY: provides sine and cosine transformations

lagged_df["hour_sin"] = np.sin(2 * np.pi * lagged_df["hour"] / 24)
lagged_df["hour_cos"] = np.cos(2 * np.pi * lagged_df["hour"] / 24)

lagged_df["day_sin"] = np.sin(2 * np.pi * lagged_df["day_of_week"] / 7)
lagged_df["day_cos"] = np.cos(2 * np.pi * lagged_df["day_of_week"] / 7)

lagged_df["month_sin"] = np.sin(2 * np.pi * lagged_df["month"] / 12)
lagged_df["month_cos"] = np.cos(2 * np.pi * lagged_df["month"] / 12)

print(lagged_df[
    ["hour", "hour_sin", "hour_cos", "day_of_week", "day_sin", "day_cos"]
].head())

   hour  hour_sin  hour_cos  day_of_week   day_sin   day_cos
0     0  0.000000  1.000000            3  0.433884 -0.900969
1     1  0.258819  0.965926            3  0.433884 -0.900969
2     2  0.500000  0.866025            3  0.433884 -0.900969
3     3  0.707107  0.707107            3  0.433884 -0.900969
4     4  0.866025  0.500000            3  0.433884 -0.900969


In [18]:
features = [
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "month_sin", "month_cos", "is_weekend",
    "price day ahead", "total load forecast",
    "forecast solar day ahead", "forecast wind onshore day ahead",
    "price_lag_1", "price_lag_2", "price_lag_3",
    "price_lag_24", "price_lag_48", "price_lag_168",
    "load_lag_1", "load_lag_24", "load_lag_168",
    "price_mean_24", "price_std_24", "load_mean_24"
]  # WHY: final leakage-safe forecasting features

X = lagged_df[features].copy()  # WHY: model inputs
y = lagged_df["price actual"].copy()  # WHY: prediction target

print("Features:", len(features))
print("X:", X.shape)
print("y:", y.shape)

Features: 23
X: (34896, 23)
y: (34896,)


In [19]:
n = len(X)  # WHY: calculate boundaries from final processed data

train_end = int(n * 0.70)  # WHY: oldest 70% trains the models
val_end = int(n * 0.85)  # WHY: following 15% tunes hyperparameters

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

print(X_train.shape, X_val.shape, X_test.shape)

(24427, 23) (5234, 23) (5235, 23)


In [20]:
from sklearn.preprocessing import StandardScaler  # WHY: normalize ANN inputs

scaler = StandardScaler()  # WHY: stores training-set mean and standard deviation

X_train_scaled = scaler.fit_transform(
    X_train
)  # WHY: learn scaling statistics ONLY from historical training data

X_val_scaled = scaler.transform(
    X_val
)  # WHY: validation data must not influence preprocessing

X_test_scaled = scaler.transform(
    X_test
)  # WHY: test set must remain completely unseen

print(X_train_scaled.shape)
print(X_val_scaled.shape)
print(X_test_scaled.shape)

(24427, 23)
(5234, 23)
(5235, 23)


In [21]:
print("Training mean:", X_train_scaled.mean())
print("Training std:", X_train_scaled.std())

print("NaN train:", np.isnan(X_train_scaled).sum())
print("NaN val:", np.isnan(X_val_scaled).sum())
print("NaN test:", np.isnan(X_test_scaled).sum())

Training mean: 2.5901337309155145e-17
Training std: 1.0
NaN train: 0
NaN val: 0
NaN test: 0


In [22]:
import os  # WHY: create folders safely
import joblib  # WHY: save the fitted scaler for future inference

os.makedirs("processed_data", exist_ok=True)  # WHY: keep processed files together

X_train.to_csv("processed_data/X_train.csv", index=False)
X_val.to_csv("processed_data/X_val.csv", index=False)
X_test.to_csv("processed_data/X_test.csv", index=False)

y_train.to_csv("processed_data/y_train.csv", index=False)
y_val.to_csv("processed_data/y_val.csv", index=False)
y_test.to_csv("processed_data/y_test.csv", index=False)

joblib.dump(scaler, "processed_data/price_scaler.pkl")  # WHY: reuse identical scaling later

['processed_data/price_scaler.pkl']